<a href="https://colab.research.google.com/github/omprakash3005/sanskrit-ocr-correction/blob/main/Sanskrit_OCR_Correction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Step 1: Install the tools we need


In [7]:
# Install OCR tool (Tesseract) and the Sanskrit language file it needs
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr
!mkdir -p /usr/share/tesseract-ocr/5/tessdata
!wget -q -O /usr/share/tesseract-ocr/5/tessdata/san.traineddata \
    https://raw.githubusercontent.com/tesseract-ocr/tessdata/main/san.traineddata

# Install Python libraries we need
!pip install -q pytesseract pillow transformers datasets accelerate sentencepiece

print("Done installing.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Done installing.


In [8]:
from google.colab import drive
drive.mount('/content/drive')

# This must point at your actual Drive folder. Based on your screenshot, this is correct:
DATA_FOLDER = "/content/drive/MyDrive/data_assign"

import os
os.environ["TESSDATA_PREFIX"] = "/usr/share/tesseract-ocr/5/tessdata"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Step 2: Load our data

We have 5 pages. Each page has:
- a picture: `page_1.png`, `page_2.png`, ...
- the correct text for that picture: `page_1.md`, `page_2.md`, ...

We will use pages 1-4 to teach the model, and save page 5 to test it at the end
(the model will never see page 5's correct text during training).


In [9]:
# Read the clean (correct) text for each page into a simple dictionary
clean_text = {}
for page_number in range(1, 6):
    file_path = f"{DATA_FOLDER}/page_{page_number}.md"
    with open(file_path, "r", encoding="utf-8") as f:
        clean_text[page_number] = f.read().strip()

print("Loaded clean text for pages:", list(clean_text.keys()))
print()
print("Example (page 1):")
print(clean_text[1])

Loaded clean text for pages: [1, 2, 3, 4, 5]

Example (page 1):
अथ आयुर्वेदः नाम शास्त्रम्
आयुर्वेदः नाम आयुः वेदयति इति आयुर्वेदः
आयुः नाम शरीर इन्द्रिय सत्त्व आत्म 
संयोगः तस्य हिताहितं सुखं दुःखम् आयुः
तस्य हितं च अहितं च मानम् च तच् च 
यत्र उक्तं तत् आयुर्वेदः


## Step 3: Run OCR on the images ("before" results)

This shows us how good (or bad) plain OCR is, before any correction.


In [10]:
import pytesseract
from PIL import Image

def run_ocr(image_path):
    """Read text out of an image using Tesseract OCR."""
    image = Image.open(image_path)
    text = pytesseract.image_to_string(image, lang="san")
    return text.strip()

ocr_text = {}
for page_number in range(1, 6):
    image_path = f"{DATA_FOLDER}/page_{page_number}.png"
    ocr_text[page_number] = run_ocr(image_path)
    print(f"--- Page {page_number}: OCR output ---")
    print(ocr_text[page_number])
    print()

--- Page 1: OCR output ---
अथ आयुर्वेदः नाम शास्त्रम्‌

आयुर्वेदः नाम आगुः ठेट्रमति इति आयुर्वेद
आरूः नाम शरीर इन्टिम सत्त्व आत्म
संयोगः तस्त्र हिताहितं युखं दृखम्‌ आयुः
तस्त्र हितं च अदितं च मानम्‌ च तच्‌ च
मत्र उक्तं तत्‌ आरव

--- Page 2: OCR output ---
त्रमो द्रोषाः वातः पित्तं कफः इति
एते द्रोणाः शरीरस धारणम्‌ कुर्वन्ति
तातः गतिः कारणम्‌

पित्तं पाक कारणम्‌

कफः स्थिरत्व कारणम्‌

--- Page 3: OCR output ---
आहारः महत्वं आयुः वर्धनम्‌ करोति
सम्यक्‌ आहारः शरीरे बलम्‌ वर्धयति
असम्यक्‌ आदारः रोग कारणम्‌ भवति
दिताहार येक्ने आगुः ठर्धमति

--- Page 4: OCR output ---
द्िनचर्मा नाम नित्य कर्म अनुष्ठानम्‌
प्रातः उत्थाने शचं दन्त धाठने कर्तव्यम्‌
व्मायामः शरीरस्य बल कर्धने करोति
स्नानं शर्खीर शुद्धि कारणम्‌

--- Page 5: OCR output ---
योगः नाम दोष वेषम्म कारणम्‌

द्रोण सम्मता आरोग्य कारणम्‌

रोग उत्पत्ति हेतवः मिथ्या आहार निहार
योग निवारणम्‌ सम्यक्‌ चिकित्या द्वारा
भवति



## Step 4: Measure how many mistakes OCR makes

We use a simple metric called **Character Error Rate (CER)**.
It counts how many single-character edits (insert / delete / change one
character) it takes to turn the OCR text into the correct text, divided by
the length of the correct text.

- CER = 0.0 means perfect (no mistakes).
- CER = 0.20 roughly means "20% of characters are wrong."


In [11]:
def edit_distance(text_a, text_b):
    """Counts the minimum number of single-character changes needed
    to turn text_a into text_b. This is the classic 'Levenshtein distance'."""
    a_len, b_len = len(text_a), len(text_b)
    # previous_row[j] = edit distance between text_a[:i] and text_b[:j]
    previous_row = list(range(b_len + 1))

    for i in range(1, a_len + 1):
        current_row = [i] + [0] * b_len
        for j in range(1, b_len + 1):
            if text_a[i - 1] == text_b[j - 1]:
                cost = 0
            else:
                cost = 1
            current_row[j] = min(
                previous_row[j] + 1,       # delete a character
                current_row[j - 1] + 1,    # insert a character
                previous_row[j - 1] + cost # keep or change a character
            )
        previous_row = current_row

    return previous_row[b_len]


def character_error_rate(correct_text, predicted_text):
    """Fraction of characters that are wrong (lower is better)."""
    if len(correct_text) == 0:
        return 0.0
    mistakes = edit_distance(correct_text, predicted_text)
    return mistakes / len(correct_text)


# Check the OCR quality on each page
print("OCR mistake rate per page (before any correction):")
for page_number in range(1, 6):
    cer = character_error_rate(clean_text[page_number], ocr_text[page_number])
    print(f"  Page {page_number}: CER = {cer:.2f}  ({cer*100:.0f}% of characters wrong)")


OCR mistake rate per page (before any correction):
  Page 1: CER = 0.19  (19% of characters wrong)
  Page 2: CER = 0.12  (12% of characters wrong)
  Page 3: CER = 0.14  (14% of characters wrong)
  Page 4: CER = 0.10  (10% of characters wrong)
  Page 5: CER = 0.14  (14% of characters wrong)


## Step 5: Create extra practice examples

We only have 5 real pages — not enough to train an AI model well. So we make
extra "practice examples" by taking clean sentences (from pages 1-4 only,
since page 5 is our test page) and randomly adding OCR-style mistakes to them.

This is like a teacher making up extra practice questions similar to a
real exam, using patterns of mistakes we've actually seen.


In [12]:
import random

# A few Devanagari characters that OCR often confuses with each other.
# We picked these by looking at the real OCR mistakes above (e.g. "वातः" became "तातः").
LOOK_ALIKE_CHARACTERS = {
    "य": "म",
    "व": "त",
    "ध": "द",
    "श": "स",
    "स": "य",
    "ह": "द",
    "े": "ॆ",
    "ो": "े",
}

VIRAMA = "्"          # a small mark under a letter, often misread by OCR
ZWNJ = "\u200c"       # an invisible character OCR sometimes mistakes it for


def add_ocr_style_mistakes(text, how_much=0.3):
    """Randomly corrupt a piece of clean text so it looks like typical OCR
    output. 'how_much' controls how many mistakes we add (0 = none, 1 = a lot)."""
    new_characters = []
    for character in text:
        roll = random.random()  # random number between 0 and 1

        if character == VIRAMA and roll < how_much:
            # OCR often misreads this mark as an invisible character
            new_characters.append(ZWNJ)
        elif character in LOOK_ALIKE_CHARACTERS and roll < how_much:
            # swap for a similar-looking character
            new_characters.append(LOOK_ALIKE_CHARACTERS[character])
        elif roll < how_much * 0.1:
            # occasionally just drop a character (faint print / smudges)
            continue
        else:
            new_characters.append(character)

    return "".join(new_characters)


# Build a list of clean sentences from pages 1-4 (page 5 is kept for testing only)
clean_sentences = []
for page_number in [1, 2, 3, 4]:
    for line in clean_text[page_number].splitlines():
        line = line.strip()
        if line:
            clean_sentences.append(line)

print(f"We have {len(clean_sentences)} clean sentences to practice with.")

# For each clean sentence, make several corrupted ("noisy") versions
training_pairs = []  # list of (noisy_text, clean_text)
for sentence in clean_sentences:
    for _ in range(15):
        mistake_amount = random.choice([0.2, 0.3, 0.4, 0.5])
        noisy_version = add_ocr_style_mistakes(sentence, how_much=mistake_amount)
        if noisy_version != sentence:
            training_pairs.append((noisy_version, sentence))

# Also add the REAL ocr mistakes from pages 1-4 as extra practice examples
for page_number in [1, 2, 3, 4]:
    training_pairs.append((ocr_text[page_number], clean_text[page_number]))

print(f"Total practice examples created: {len(training_pairs)}")
print()
print("Example practice pair:")
print("  Noisy :", training_pairs[0][0])
print("  Clean :", training_pairs[0][1])


We have 19 clean sentences to practice with.
Total practice examples created: 266

Example practice pair:
  Noisy : अथ आयर्वेदः नाम शास्त्रम्
  Clean : अथ आयुर्वेदः नाम शास्त्रम्


## Step 6: Split into training and validation sets

We keep a small slice of our practice examples aside (validation set) so we
can check the model isn't just memorizing.


In [13]:
random.shuffle(training_pairs)

number_for_validation = max(1, int(len(training_pairs) * 0.1))
validation_pairs = training_pairs[:number_for_validation]
train_pairs = training_pairs[number_for_validation:]

print(f"Training examples:   {len(train_pairs)}")
print(f"Validation examples: {len(validation_pairs)}")


Training examples:   240
Validation examples: 26


## Step 7: Train the AI model

We use a small, ready-made model called **ByT5-small** and teach it to turn
noisy text into clean text. We picked ByT5 because it reads text
letter-by-letter (byte by byte) instead of using a fixed dictionary of word
pieces — that matters here because OCR mistakes often create unusual
character combinations a normal dictionary wouldn't recognize.

This step trains for a few minutes on a Colab GPU.


In [15]:
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
from datasets import Dataset

MODEL_NAME = "google/byt5-small"
INSTRUCTION = "correct sanskrit ocr: "  # tells the model what task to do

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)


def prepare_examples(pairs):
    """Turn our (noisy, clean) text pairs into a format the model can train on."""
    noisy_texts = [INSTRUCTION + noisy for noisy, clean in pairs]
    clean_texts = [clean for noisy, clean in pairs]

    dataset = Dataset.from_dict({"noisy": noisy_texts, "clean": clean_texts})

    def tokenize(batch):
        model_inputs = tokenizer(batch["noisy"], max_length=256, truncation=True)
        labels = tokenizer(text_target=batch["clean"], max_length=256, truncation=True)
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

    return dataset.map(tokenize, batched=True, remove_columns=dataset.column_names)


train_dataset = prepare_examples(train_pairs)
validation_dataset = prepare_examples(validation_pairs)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_settings = Seq2SeqTrainingArguments(
    output_dir="ocr_correction_model",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,          # how many times to go through all the training data
    learning_rate=3e-4,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    predict_with_generate=True,
    fp16=True,                    # faster training on GPU
    logging_steps=10,
    report_to=[],
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_settings,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,   # <-- renamed from `tokenizer=` in newer transformers versions
)

trainer.train()

# Save the trained model so we can use it later
trainer.save_model("ocr_correction_model")
tokenizer.save_pretrained("ocr_correction_model")
print("Training finished. Model saved to 'ocr_correction_model'.")


Loading weights:   0%|          | 0/172 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Map:   0%|          | 0/240 [00:00<?, ? examples/s]

Map:   0%|          | 0/26 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss
1,0.000000,nan
2,0.000000,nan
3,0.000000,nan
4,0.000000,nan
5,0.000000,nan
6,0.000000,nan
7,0.000000,nan
8,0.000000,nan
9,0.000000,nan
10,0.000000,nan


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training finished. Model saved to 'ocr_correction_model'.


## Step 8: Test the model on page 5 (never seen before)

Now we check: did the model actually learn to fix OCR mistakes, or not?
We compare the mistake rate (CER) **before** correction (raw OCR) and
**after** correction (model output), on page 5 — which the model never saw
during training.


In [20]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

def correct_text(noisy_text):
    """Feed one line of noisy OCR text to the model and get the corrected version."""
    inputs = tokenizer(INSTRUCTION + noisy_text, return_tensors="pt", truncation=True, max_length=256).to(device)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_length=256, num_beams=4)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

page5_ocr_lines = [line for line in ocr_text[5].splitlines() if line.strip()]
corrected_lines = [correct_text(line) for line in page5_ocr_lines]
corrected_page5 = "\n".join(corrected_lines)

print("--- Page 5: raw OCR (before correction) ---")
print(ocr_text[5])
print()
print("--- Page 5: model output (after correction) ---")
print(corrected_page5)
print()
print("--- Page 5: ground truth (correct answer) ---")
print(clean_text[5])
print()

cer_before = character_error_rate(clean_text[5], ocr_text[5])
cer_after = character_error_rate(clean_text[5], corrected_page5)
print(f"CER before correction: {cer_before:.2f}")
print(f"CER after correction:  {cer_after:.2f}")

--- Page 5: raw OCR (before correction) ---
योगः नाम दोष वेषम्म कारणम्‌

द्रोण सम्मता आरोग्य कारणम्‌

रोग उत्पत्ति हेतवः मिथ्या आहार निहार
योग निवारणम्‌ सम्यक्‌ चिकित्या द्वारा
भवति

--- Page 5: model output (after correction) ---
: योगः नाम दोष वेषम्‌ कारणम्‌ कारणम्‌ कारणम्‌ कारणम्‌ कारणम्‌ कारणम्
: द्रोणसम्மता आरोग्य कारणम्‌ आरोग्य कारणम्‌ आरोग्य कारणम्‌ आरोग्य क 
: रोग उत्पत्ति हेतवः मिथ्या आहार निहर: रोग उत्पत्ति हेतवः मिथ्या आहार
: योग निवारणम्‌ सम्यक्‌ चिकित्‌ चिकित्‌ चिकित्‌ चिकित्‌ चिकित्‌ चिकि
: भवति
भवति
भवति
भवति
भवति
भवति
भवति
भवति
भवति
भवति
भवति
भवति
भवति
भवति

--- Page 5: ground truth (correct answer) ---
रोगः नाम दोष वैषम्य कारणम्
दोष सम्यता आरोग्य कारणम्
रोग उत्पत्ति हेतवः मिथ्या आहार विहारः
रोग निवारणम् सम्यक् चिकित्सा द्वारा भवति

CER before correction: 0.14
CER after correction:  1.80


## Step 9 (optional): Try it on your own text

Paste in any noisy Sanskrit OCR text and see what the model produces.


In [21]:
my_text = "मत्र उक्तं तत्‌ आयुर्वेद"   # <-- replace with your own noisy text
print(correct_text(my_text))

: उक्तं तततततततततततततततततततततततततततततततततततततततततततततततततततत
